# 08 Sensitivity Checks

**Objective:** test whether the main findings (strong positive Global Moran's I; seven Manhattan High-High clusters in all four years; Jackson Heights Low-Low in all four years; widespread rate increases 2022 to 2025) depend on analytical choices.

**Approach:** one function repeats the full pipeline (two-step neighbourhood medians, Queen weights, island removal, Global Moran's I, LISA with 5% Benjamini-Hochberg FDR) under alternative choices. The function is first run on the unchanged data to check that it reproduces notebook 06.

In [1]:
from pathlib import Path
import warnings
import pandas as pd
import numpy as np
import geopandas as gpd
from libpysal import weights
from esda.moran import Moran, Moran_Local

pd.set_option("display.width", 150)
pd.set_option("display.max_rows", 200)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, PROCESSED, TAB = PROJECT_ROOT / "data" / "raw", PROJECT_ROOT / "data" / "processed", PROJECT_ROOT / "outputs" / "tables"

YEARS, ALPHA = [2022, 2023, 2024, 2025], 0.05
QUAD = {1: "High-High", 2: "Low-High", 3: "Low-Low", 4: "High-Low"}
CORE_HH = ["Chelsea-Hudson Yards", "Gramercy", "Greenwich Village", "Midtown South-Flatiron-Union Square",
           "Midtown-Times Square", "Murray Hill-Kips Bay", "West Village"]

df = pd.read_parquet(PROCESSED / "past_rates_clean.parquet")
nta = gpd.read_file(RAW / "nta2020.geojson")[["nta2020", "ntaname", "boroname", "ntatype", "geometry"]].to_crs(2263)
nta_res = nta[nta["ntatype"] == "0"]

def bh_threshold(p, alpha=0.05):
    p = np.sort(np.asarray(p)); m = len(p)
    ok = np.where(p <= alpha * np.arange(1, m + 1) / m)[0]
    return p[ok.max()] if len(ok) else 0.0

def nta_medians(data, years, min_listings):
    """Step 1: median per listing-year. Step 2: median per NTA-year. Keep NTAs with enough listings."""
    d = data[data["use_in_spatial"] & data["year"].isin(years)]
    ly = d.groupby(["nta2020", "listing_id", "year"])["avg_daily_rate"].median().rename("listing_median").reset_index()
    ny = (ly.groupby(["nta2020", "year"])
            .agg(median_rate=("listing_median", "median"), n_listings=("listing_id", "nunique")).reset_index())
    return ny[ny["n_listings"] >= min_listings]

def run_pipeline(data, years=YEARS, min_listings=5, seed=2026):
    elig = nta_medians(data, years, min_listings)
    rows, labels = [], {}
    for y in years:
        g = nta_res.merge(elig.loc[elig["year"] == y, ["nta2020", "median_rate", "n_listings"]], on="nta2020").reset_index(drop=True)
        row = {"year": y, "eligible": len(g)}
        if len(g) < 20:                                     # too few NTAs for a meaningful analysis
            rows.append({**row, "note": "too few NTAs"}); continue
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            w = weights.Queen.from_dataframe(g)
        g2 = g[~g.index.isin(list(w.islands))].reset_index(drop=True)      # drop islands (no neighbours)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            w2 = weights.Queen.from_dataframe(g2)
        w2.transform = "r"
        vals = g2["median_rate"].values
        np.random.seed(seed)
        gm = Moran(vals, w2, permutations=999)
        ml = Moran_Local(vals, w2, permutations=999, seed=seed)
        quad = pd.Series(ml.q).map(QUAD).values
        g2["cluster"] = np.where(ml.p_sim <= bh_threshold(ml.p_sim, ALPHA), quad, "Not significant")
        labels[y] = g2[["nta2020", "ntaname", "boroname", "median_rate", "n_listings", "cluster"]]
        c = g2["cluster"]
        row.update({"analysed": len(g2), "Moran_I": gm.I, "p_value": gm.p_sim,
                    "HH": int((c == "High-High").sum()), "LL": int((c == "Low-Low").sum()),
                    "HL": int((c == "High-Low").sum()), "LH": int((c == "Low-High").sum()),
                    "core_HH_of_7": int(g2.loc[c == "High-High", "ntaname"].isin(CORE_HH).sum()),
                    "JacksonHts_LL": bool(((g2["ntaname"] == "Jackson Heights") & (c == "Low-Low")).any())})
        rows.append(row)
    return rows, labels

print("Rows:", len(df), "| residential NTAs:", len(nta_res))

Rows: 150034 | residential NTAs: 197


baseline test (must reproduce notebook 06)

In [2]:
base_rows, base_labels = run_pipeline(df)
base = pd.DataFrame(base_rows)
print(base.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# Compare with the values reported in notebook 06 (used only as a check, not as output)
ref = {"analysed": [124, 119, 120, 94], "HH": [14, 14, 13, 7], "LL": [2, 3, 4, 4], "Moran_I": [0.685, 0.687, 0.650, 0.569]}
same = (list(base["analysed"]) == ref["analysed"] and list(base["HH"]) == ref["HH"] and list(base["LL"]) == ref["LL"]
        and np.allclose(base["Moran_I"].round(3), ref["Moran_I"]))
print("\nReproduces notebook 06 (analysed NTAs, HH, LL, Moran's I):", same)

 year  eligible  analysed  Moran_I  p_value  HH  LL  HL  LH  core_HH_of_7  JacksonHts_LL
 2022       129       124    0.685    0.001  14   2   0   0             7           True
 2023       123       119    0.687    0.001  14   3   0   0             7           True
 2024       125       120    0.650    0.001  13   4   0   0             7           True
 2025        98        94    0.569    0.001   7   4   0   1             7           True

Reproduces notebook 06 (analysed NTAs, HH, LL, Moran's I): True


## 1. Minimum-listings threshold (3 and 10 versus the baseline of 5)

In [3]:
all_rows, all_labels = [], {}

def add_scenario(name, data, **kw):
    r, l = run_pipeline(data, **kw)
    for x in r:
        x["scenario"] = name
    all_rows.extend(r); all_labels[name] = l

add_scenario("baseline (min 5)", df)
add_scenario("min 3 listings", df, min_listings=3)
add_scenario("min 10 listings", df, min_listings=10)

cols = ["scenario", "year", "eligible", "analysed", "Moran_I", "p_value", "HH", "LL", "HL", "LH", "core_HH_of_7", "JacksonHts_LL"]
tbl = pd.DataFrame(all_rows)
print(tbl[[c for c in cols if c in tbl.columns]].to_string(index=False, float_format=lambda v: f"{v:.3f}"))

for sc in ["min 3 listings", "min 10 listings"]:
    for y, lab in all_labels[sc].items():
        ll = lab.loc[lab["cluster"] == "Low-Low", "ntaname"].tolist()
        hh_other = lab.loc[(lab["cluster"] == "High-High") & ~lab["ntaname"].isin(CORE_HH), "ntaname"].tolist()
        print(f"\n{sc}, {y}\n  Low-Low: {'; '.join(ll) or '-'}\n  High-High outside the core seven: {'; '.join(hh_other) or '-'}")

        scenario  year  eligible  analysed  Moran_I  p_value  HH  LL  HL  LH  core_HH_of_7  JacksonHts_LL
baseline (min 5)  2022       129       124    0.685    0.001  14   2   0   0             7           True
baseline (min 5)  2023       123       119    0.687    0.001  14   3   0   0             7           True
baseline (min 5)  2024       125       120    0.650    0.001  13   4   0   0             7           True
baseline (min 5)  2025        98        94    0.569    0.001   7   4   0   1             7           True
  min 3 listings  2022       149       145    0.639    0.001  14   5   1   0             6           True
  min 3 listings  2023       146       142    0.655    0.001  15   5   0   0             7           True
  min 3 listings  2024       152       148    0.608    0.001  14   5   0   0             7           True
  min 3 listings  2025       130       126    0.522    0.001  12   5   0   1             7           True
 min 10 listings  2022        87        82    

### Which core High-High neighbourhoods are missing in some scenarios, and why?
For each non-baseline scenario and year in which fewer than seven core neighbourhoods are High-High, we list the missing ones and whether they were not analysed (too few listings) or analysed but not High-High.

In [4]:
for sc in ["min 3 listings", "min 10 listings"]:
    for y, lab in all_labels[sc].items():
        hh = set(lab.loc[lab["cluster"] == "High-High", "ntaname"])
        missing = [n for n in CORE_HH if n not in hh]
        if missing:
            parts = []
            for n in missing:
                r = lab[lab["ntaname"] == n]
                parts.append(f"{n} (not analysed)" if r.empty else f"{n} ({r['cluster'].iloc[0]}, {int(r['n_listings'].iloc[0])} listings)")
            print(f"{sc}, {y}: " + "; ".join(parts))

pd.DataFrame(all_rows).to_csv(TAB / "08_sensitivity_thresholds.csv", index=False)
print("\nSaved 08_sensitivity_thresholds.csv")

min 3 listings, 2022: Gramercy (Not significant, 33 listings)
min 10 listings, 2022: Gramercy (Not significant, 33 listings)
min 10 listings, 2025: Gramercy (Not significant, 24 listings)

Saved 08_sensitivity_thresholds.csv


## 2. Occupancy above 0 only, and entire homes only
The full pipeline is repeated (a) using only months with occupancy above 0, and (b) using only entire homes. The minimum of 5 listings per NTA is kept, so NTAs with fewer qualifying listings drop out of these scenarios.

In [5]:
rows2, labels2 = [], {}
for name, data in [("occupancy > 0 only", df[df["occupancy"] > 0]),
                   ("entire homes only", df[df["room_type"] == "entire_home"])]:
    r, l = run_pipeline(data)
    for x in r:
        x["scenario"] = name
    rows2.extend(r); labels2[name] = l

t2 = pd.DataFrame(rows2)
print(t2[[c for c in cols if c in t2.columns]].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
t2.to_csv(TAB / "08_sensitivity_occupancy_roomtype.csv", index=False)

for name, labs in labels2.items():
    for y, lab in labs.items():
        print(f"\n{name}, {y}  Low-Low: {'; '.join(lab.loc[lab['cluster'] == 'Low-Low', 'ntaname']) or '-'}")

          scenario  year  eligible  analysed  Moran_I  p_value  HH  LL  HL  LH  core_HH_of_7  JacksonHts_LL
occupancy > 0 only  2022       126       120    0.662    0.001   9   2   0   0             6           True
occupancy > 0 only  2023       114       108    0.611    0.001   8   2   0   1             6           True
occupancy > 0 only  2024        85        82    0.532    0.001   2   2   0   1             2           True
occupancy > 0 only  2025        73        71    0.569    0.001   4   2   0   0             4           True
 entire homes only  2022        86        80    0.358    0.001   6   2   1   1             4          False
 entire homes only  2023        84        78    0.368    0.001   0   0   0   0             0          False
 entire homes only  2024        93        86    0.353    0.001   1   1   1   0             1          False
 entire homes only  2025        68        61    0.362    0.002   0   0   0   0             0          False

occupancy > 0 only, 2022  L

## 3. Listings present in both 2022 and 2025
The 2022-to-2025 change in neighbourhood median rate is recalculated using only listings that have data in both years, so that changes cannot come from listings entering or leaving the data. Only NTAs with at least 5 such listings are included, and the result is compared with the baseline change (notebook 04).

In [6]:
ly = (df[df["use_in_spatial"] & df["year"].isin([2022, 2025])]
      .groupby(["nta2020", "listing_id", "year"])["avg_daily_rate"].median().rename("m").reset_index())
wide = ly.pivot(index=["nta2020", "listing_id"], columns="year", values="m").dropna().reset_index()
wide = wide.rename(columns={2022: "r22", 2025: "r25"})
print("Listings with data in both 2022 and 2025:", len(wide), "of", ly["listing_id"].nunique())

g = wide.groupby("nta2020").agg(rate_2022=("r22", "median"), rate_2025=("r25", "median"), n_both=("listing_id", "nunique"))
g = g[g["n_both"] >= 5].copy()
g["pct_same_listings"] = 100 * (g["rate_2025"] / g["rate_2022"] - 1)

b = nta_medians(df, [2022, 2025], 5).pivot(index="nta2020", columns="year", values="median_rate").dropna()
b["pct_baseline"] = 100 * (b[2025] / b[2022] - 1)
print("Baseline check, NTAs eligible in both years:", len(b), "| median change:", round(b["pct_baseline"].median(), 1), "(notebook 04: 96 NTAs, +17.7)")

cmp = g.join(b[["pct_baseline"]], how="inner")
print("\nSame-listings analysis: NTAs with at least 5 listings in both years:", len(g), "| also in baseline:", len(cmp))
print("Median change:", round(g["pct_same_listings"].median(), 1), "| mean:", round(g["pct_same_listings"].mean(), 1))
print("NTAs rising / falling:", int((g["pct_same_listings"] > 0).sum()), "/", int((g["pct_same_listings"] < 0).sum()))
print("Same direction as baseline:", int(((cmp["pct_same_listings"] > 0) == (cmp["pct_baseline"] > 0)).sum()), "of", len(cmp))
print("Correlation with baseline change:", round(cmp["pct_same_listings"].corr(cmp["pct_baseline"]), 2))
g.reset_index().to_csv(TAB / "08_pct_change_same_listings.csv", index=False)

Listings with data in both 2022 and 2025: 1392 of 4221
Baseline check, NTAs eligible in both years: 96 | median change: 17.7 (notebook 04: 96 NTAs, +17.7)

Same-listings analysis: NTAs with at least 5 listings in both years: 79 | also in baseline: 79
Median change: 2.6 | mean: 3.8
NTAs rising / falling: 48 / 31
Same direction as baseline: 48 of 79
Correlation with baseline change: 0.03


why jackson heights and the manhattan core change in the entire-homes run

In [7]:
core_plus = CORE_HH + ["Jackson Heights", "Woodside"]
ids = nta_res.loc[nta_res["ntaname"].isin(core_plus), ["nta2020", "ntaname"]]

d = df[df["use_in_spatial"] & df["year"].isin(YEARS) & df["nta2020"].isin(ids["nta2020"])]
lst = d.drop_duplicates("listing_id")[["listing_id", "nta2020", "room_type"]].merge(ids, on="nta2020")
comp = pd.crosstab(lst["ntaname"], lst["room_type"])
comp["% entire home"] = (100 * comp["entire_home"] / comp.sum(axis=1)).round(0)

eh = nta_medians(df[df["room_type"] == "entire_home"], YEARS, 1)      # min 1 so we can see small counts
cnt = eh.pivot(index="nta2020", columns="year", values="n_listings")
out = ids.set_index("nta2020").join(cnt).set_index("ntaname").add_prefix("EH listings ")
print("Listings by room type (all years) and share of entire homes:")
print(comp.to_string())
print("\nEntire-home listings per year (blank = none; at least 5 are needed to be analysed):")
print(out.drop(columns=[c for c in out.columns if c.endswith("nta2020")], errors="ignore").to_string())

Listings by room type (all years) and share of entire homes:
room_type                            entire_home  hotel_room  private_room  shared_room  % entire home
ntaname                                                                                               
Chelsea-Hudson Yards                          70           3            31            1           67.0
Gramercy                                      29           0            10            0           74.0
Greenwich Village                             16           0             3            0           84.0
Jackson Heights                               14           0            25            0           36.0
Midtown South-Flatiron-Union Square           54           2            41            2           55.0
Midtown-Times Square                          51           5            48            1           49.0
Murray Hill-Kips Bay                          54           0             7            1           87.0
West Village

the citywide change on the same listings

In [8]:
both_ids = wide["listing_id"]                         # listings with data in both 2022 and 2025 (from the previous cell)
c = df[df["listing_id"].isin(both_ids) & df["year"].isin([2022, 2025])]
l = c.groupby(["listing_id", "year"])["avg_daily_rate"].median().unstack().dropna()
print("Listings:", len(l))
print("Median of listing-level medians:", l.median().round(1).to_dict())
print("Citywide change in that median:", round(100 * (l[2025].median() / l[2022].median() - 1), 1), "%")
print("Median of each listing's own % change:", round((100 * (l[2025] / l[2022] - 1)).median(), 1), "%")
print("Share of these listings with a higher rate in 2025:", round(100 * (l[2025] > l[2022]).mean(), 1), "%")

Listings: 1392
Median of listing-level medians: {2022: 140.9, 2025: 149.6}
Citywide change in that median: 6.2 %
Median of each listing's own % change: 5.0 %
Share of these listings with a higher rate in 2025: 62.4 %


In [9]:
comp.reset_index().to_csv(TAB / "08_roomtype_mix_core_nta.csv", index=False)
pd.DataFrame({"statistic": ["listings_both_years", "median_2022", "median_2025", "pct_change_median",
                            "median_own_pct_change", "share_higher_in_2025"],
              "value": [len(l), l[2022].median(), l[2025].median(),
                        100 * (l[2025].median() / l[2022].median() - 1),
                        (100 * (l[2025] / l[2022] - 1)).median(),
                        100 * (l[2025] > l[2022]).mean()]}).round(2).to_csv(TAB / "08_same_listings_citywide.csv", index=False)
print("Saved diagnostics")

Saved diagnostics


## Summary
- The pipeline function reproduces notebook 06 exactly (analysed NTAs, High-High and Low-Low counts and Global Moran's I for 2022-2025).
- **Minimum-listings threshold (3, 5, 10):** Global Moran's I stayed strongly positive (0.522 to 0.759, p = 0.001 in all 12 runs); Jackson Heights was Low-Low in all 12; all seven core Manhattan neighbourhoods were High-High in 9 of 12 runs, and in the other three Gramercy was analysed but not significant. High-High neighbourhoods fell from 2022 to 2025 by 14 to 12 (min 3), 14 to 7 (min 5) and 10 to 6 (min 10), so the 2025 reduction is threshold-dependent. Most extra Low-Low neighbourhoods appeared under one threshold only.
- **Months with occupancy above 0:** Moran's I 0.662, 0.611, 0.532, 0.569; Woodside and Jackson Heights Low-Low in all four years; Manhattan core High-High count 6, 6, 2, 4 of 7.
- **Entire homes only:** Moran's I about 0.36 in every year (p <= 0.002); High-High 6, 0, 1, 0; Low-Low 2, 0, 1, 0; Jackson Heights not a Low-Low cluster in any year although it had at least 5 entire-home listings each year. Entire homes made up 49-93% of listings in the core Manhattan neighbourhoods and 36% (Jackson Heights) and 24% (Woodside) in the Low-Low neighbourhoods.
- **Listings present in both 2022 and 2025 (1,392 of 4,221):** median rate $140.9 to $149.6 (+6.2%); 62.4% of listings had a higher rate in 2025. At neighbourhood level (79 NTAs with at least 5 such listings) the median change was +2.6% and agreed in direction with the baseline change in only 48 of 79 NTAs (r = 0.03).

## Limitations / observations
- Spatial autocorrelation is robust; the specific membership of clusters, the 2025 reduction in High-High clusters and the Chinatown-Two Bridges outlier label are not.
- Part of the clustering reflects the mix of room types across neighbourhoods.
- Neighbourhood-level changes between 2022 and 2025 are not robust and should not be read as price movements; same-listing changes are much smaller than all-listing changes. The same-listings group may differ from listings that left the data. All values are nominal USD.